
# ArXiv Filter F1 Score Evaluation

This notebook evaluates the accuracy of the ArXiv paper classification filter using F1 score and other metrics.

## Overview
- Load ground truth data from text files (positive and negative examples)
- Fetch paper metadata for all arXiv IDs
- Run classification using the ArXivClassifier
- Calculate comprehensive evaluation metrics including F1 score
- Analyze results and identify failure modes

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
from datetime import datetime
import arxiv
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score, accuracy_score

# Import our classification module
from arxiv_tools.arxiv_filter import ArXivClassifier

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print(" Imports successful")
print(f"Current working directory: {os.getcwd()}")

 Imports successful
Current working directory: /Users/tobig/code/MasterThesis/OCR/code


/Users/tobig/code/MasterThesis/OCR/code/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Set up file paths and evaluation parameters.

In [2]:
# Configuration parameters
CONFIG = {
    # File paths for ground truth data
    'positive_samples_file': 'data/arxiv_ids/arxivIDs_clean.txt',  # File containing actual model arXiv IDs
    'negative_samples_file': 'data/arxiv_ids/arxivIDs_random_500.txt',  # File containing random arXiv IDs

    # Classification parameters
    'provider': 'gemini',  # LLM provider: 'deepseek' or 'gemini'
    'confidence_threshold': 0.1,  # Confidence threshold for positive classification
    'rate_limit_delay': 0.05,  # Delay between API calls
    
    # Evaluation parameters
    'test_different_thresholds': False,  # Whether to test multiple confidence thresholds
    'threshold_range': np.arange(0.1, 1.0, 0.1),  # Range of thresholds to test
    
    # Output settings
    'save_results': True,
    'results_dir': 'results/evaluation/',
    'plot_results': True
}

# Create results directory if it doesn't exist
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(" Configuration set up:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

 Configuration set up:
   positive_samples_file: data/arxiv_ids/arxivIDs_clean.txt
   negative_samples_file: data/arxiv_ids/arxivIDs_random_500.txt
   provider: gemini
   confidence_threshold: 0.1
   rate_limit_delay: 0.05
   test_different_thresholds: False
   threshold_range: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
   save_results: True
   results_dir: results/evaluation/
   plot_results: True


## Data Loading Functions

Functions to load ground truth data and fetch paper metadata from arXiv.

In [3]:
def load_arxiv_ids(filepath: str) -> List[str]:
    """
    Load arXiv IDs from a text file.
    
    Args:
        filepath: Path to text file containing arXiv IDs (one per line)
        
    Returns:
        List of arXiv IDs
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            # Read lines, strip whitespace, and filter out empty lines
            arxiv_ids = [line.strip() for line in f.readlines() if line.strip()]
        
        print(f" Loaded {len(arxiv_ids)} arXiv IDs from {filepath}")
        return arxiv_ids
    
    except FileNotFoundError:
        print(f" File not found: {filepath}")
        print(f"   Please create this file with arXiv IDs (one per line)")
        return []
    except Exception as e:
        print(f" Error loading {filepath}: {e}")
        return []

# Test the function (will show file not found until you add the files)
positive_ids = load_arxiv_ids(CONFIG['positive_samples_file'])
negative_ids = load_arxiv_ids(CONFIG['negative_samples_file'])

print(f"\n Data Summary:")
print(f"   Positive samples (robotics): {len(positive_ids)}")
print(f"   Negative samples (non-robotics): {len(negative_ids)}")
print(f"   Total samples: {len(positive_ids) + len(negative_ids)}")

 Loaded 253 arXiv IDs from data/arxiv_ids/arxivIDs_clean.txt
 Loaded 228 arXiv IDs from data/arxiv_ids/arxivIDs_random_500.txt

 Data Summary:
   Positive samples (robotics): 253
   Negative samples (non-robotics): 228
   Total samples: 481


In [4]:
def is_valid_arxiv_id(arxiv_id: str) -> bool:
    """
    Check if a string is a valid arXiv ID format.
    
    Args:
        arxiv_id: String to check
        
    Returns:
        Boolean indicating if it's a valid arXiv ID
    """
    import re
    
    # ArXiv ID patterns
    # Old format: subject-class/YYMMnnn or subject-class/YYMMnnnvN
    # New format: YYMM.nnnn or YYMM.nnnnvN  
    patterns = [
        r'^[a-z-]+(\.[A-Z]{2})?/\d{7}(v\d+)?$',  # Old format
        r'^\d{4}\.\d{4,5}(v\d+)?$'                # New format  
    ]
    
    for pattern in patterns:
        if re.match(pattern, arxiv_id):
            return True
    return False

def clean_arxiv_ids(arxiv_ids: List[str]) -> Tuple[List[str], List[str]]:
    """
    Clean and validate arXiv IDs, separating valid from invalid ones.
    
    Args:
        arxiv_ids: List of arXiv ID strings
        
    Returns:
        Tuple of (valid_ids, invalid_ids)
    """
    valid_ids = []
    invalid_ids = []
    
    for arxiv_id in arxiv_ids:
        # Clean the ID (remove whitespace, common prefixes)
        cleaned_id = arxiv_id.strip()
        
        # Remove common prefixes if present
        if cleaned_id.startswith('arxiv:'):
            cleaned_id = cleaned_id[6:]
        if cleaned_id.startswith('arXiv:'):
            cleaned_id = cleaned_id[6:]
            
        # Skip obviously invalid IDs
        if any(invalid_pattern in cleaned_id.lower() for invalid_pattern in 
               ['http', 'www', '.html', '.com', 'jamba', 'seamless', 'cogvlm', 'llava', 'imagebi', 'bloomz', 'blip', 'fingpt']):
            invalid_ids.append(arxiv_id)
            continue
            
        # Check if it's a valid arXiv ID
        if is_valid_arxiv_id(cleaned_id):
            valid_ids.append(cleaned_id)
        else:
            invalid_ids.append(arxiv_id)
    
    return valid_ids, invalid_ids

def fetch_paper_metadata(arxiv_ids: List[str], batch_size: int = 50) -> Tuple[List[Dict], List[str]]:
    """
    Fetch paper metadata (title, abstract) from arXiv API using the new Client interface.
    
    Args:
        arxiv_ids: List of arXiv IDs
        batch_size: Number of papers to fetch per API call (reduced for stability)
        
    Returns:
        Tuple of (papers, failed_ids)
    """
    import time
    
    # Clean and validate IDs first
    valid_ids, invalid_ids = clean_arxiv_ids(arxiv_ids)
    
    print(f" Fetching metadata for {len(arxiv_ids)} total IDs...")
    print(f"   Valid arXiv IDs: {len(valid_ids)}")
    print(f"   Invalid/skipped IDs: {len(invalid_ids)}")
    
    if invalid_ids:
        print(f"   First few invalid IDs: {invalid_ids[:5]}")
    
    papers = []
    failed_ids = list(invalid_ids)  # Start with invalid IDs as failed
    
    if not valid_ids:
        print("️ No valid arXiv IDs found to fetch")
        return papers, failed_ids
    
    # Initialize the new arxiv client
    client = arxiv.Client()
    
    # Process in smaller batches to avoid API limits
    for i in range(0, len(valid_ids), batch_size):
        batch_ids = valid_ids[i:i + batch_size]
        batch_num = i // batch_size + 1
        total_batches = (len(valid_ids) - 1) // batch_size + 1
        
        print(f"   Processing batch {batch_num}/{total_batches} ({len(batch_ids)} papers)")
        
        try:
            # Create search with ID list using new API
            search = arxiv.Search(id_list=batch_ids)
            
            # Fetch results using the new client interface
            results = client.results(search)
            
            batch_papers = []
            for paper in results:
                # Extract clean arXiv ID from the entry_id URL
                paper_id = paper.entry_id.split('/')[-1]
                # Remove version number if present for matching
                clean_paper_id = paper_id.split('v')[0] if 'v' in paper_id else paper_id
                
                batch_papers.append({
                    'arxiv_id': paper_id,
                    'title': paper.title.replace('\n', ' ').strip(),
                    'abstract': paper.summary.replace('\n', ' ').strip(),
                    'authors': [str(author) for author in paper.authors],
                    'published': paper.published.isoformat() if paper.published else None,
                    'categories': paper.categories
                })
            
            papers.extend(batch_papers)
            print(f"       Fetched {len(batch_papers)} papers")
            
            # Add small delay between batches to be respectful to the API
            if batch_num < total_batches:
                time.sleep(0.5)
                
        except Exception as e:
            print(f"      ️ Error fetching batch: {e}")
            failed_ids.extend(batch_ids)
            continue
    
    # Check for missing papers by comparing fetched IDs with requested IDs
    fetched_ids = {paper['arxiv_id'].split('v')[0] for paper in papers}  # Remove version for comparison
    requested_ids = {id.split('v')[0] for id in valid_ids}  # Remove version for comparison
    missing_ids = requested_ids - fetched_ids
    
    if missing_ids:
        print(f"   ️ Could not fetch {len(missing_ids)} valid papers")
        failed_ids.extend(missing_ids)
    
    print(f" Successfully fetched {len(papers)}/{len(arxiv_ids)} total papers")
    print(f"   Valid papers fetched: {len(papers)}/{len(valid_ids)}")
    
    return papers, failed_ids

# Fetch metadata for all papers (this may take a while)
if positive_ids or negative_ids:
    all_ids = positive_ids + negative_ids
    print(f" Starting to fetch metadata for {len(all_ids)} papers...")
    all_papers, failed_ids = fetch_paper_metadata(all_ids)
    
    # Create ground truth labels using flexible matching
    paper_labels = {}
    for paper in all_papers:
        paper_arxiv_id = paper['arxiv_id']
        clean_paper_id = paper_arxiv_id.split('v')[0]  # Remove version for matching
        
        # Check against positive IDs (with and without versions)
        is_positive = False
        for pos_id in positive_ids:
            clean_pos_id = pos_id.split('v')[0]
            if clean_paper_id == clean_pos_id or paper_arxiv_id == pos_id:
                is_positive = True
                break
        
        if is_positive:
            paper_labels[paper_arxiv_id] = 1  # Positive (robotics)
        else:
            # Check against negative IDs  
            is_negative = False
            for neg_id in negative_ids:
                clean_neg_id = neg_id.split('v')[0]
                if clean_paper_id == clean_neg_id or paper_arxiv_id == neg_id:
                    is_negative = True
                    break
            
            if is_negative:
                paper_labels[paper_arxiv_id] = 0  # Negative (non-robotics)
    
    print(f"\n Metadata Summary:")
    print(f"   Successfully fetched: {len(all_papers)} papers")
    print(f"   Failed to fetch: {len(failed_ids)} papers")
    print(f"   Positive labels: {sum(paper_labels.values())}")
    print(f"   Negative labels: {len(paper_labels) - sum(paper_labels.values())}")
    print(f"   Papers without labels: {len(all_papers) - len(paper_labels)}")
    
    if len(failed_ids) > 0:
        print(f"\n️ Failed to fetch papers. Common reasons:")
        print(f"   - Invalid arXiv ID format")
        print(f"   - Paper doesn't exist or was withdrawn")  
        print(f"   - API rate limiting or connection issues")
        
else:
    print("️ No arXiv IDs loaded. Please add the text files first.")
    all_papers = []
    paper_labels = {}

2025-10-04 17:34:50,581 - arxiv - INFO - Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=&id_list=2412.04431v1%2C2409.11402%2C2410.06158v1%2C2406.09246%2C2407.15390%2C2407.15390%2C2403.20329%2C2403.09611%2C2402.15627%2C2402.07827%2C2210.16257%2C2404.01657%2C2312.08914%2C2312.06674%2C2312.00752%2C2311.07919%2C2311.07362%2C2311.07575%2C2311.04257%2C2311.05640%2C2310.19341%2C2310.17347v2%2C2310.08864%2C2310.07704%2C2309.05665%2C2308.16149%2C2308.12966%2C2307.15818%2C2209.03143%2C2306.15794%2C2306.11706%2C2306.05284%2C2305.18565%2C2305.14201%2C2305.07922%2C2305.11172v1%2C2305.09617%2C2305.06500%2C2305.06161%2C2304.08485%2C2304.07193%2C2204.05999%2C2303.17564%2C2303.16727v2%2C2303.10845%2C2303.03378%2C2209.15352%2C2212.09748%2C2302.06675v4%2C2302.05442v1&sortBy=relevance&sortOrder=descending&start=0&max_results=100


 Starting to fetch metadata for 481 papers...
 Fetching metadata for 481 total IDs...
   Valid arXiv IDs: 480
   Invalid/skipped IDs: 1
   First few invalid IDs: ['ImageBind']
   Processing batch 1/10 (50 papers)


2025-10-04 17:34:51,386 - arxiv - INFO - Got first page: 50 of 50 total results


       Fetched 50 papers


2025-10-04 17:34:51,893 - arxiv - INFO - Sleeping: 2.422157 seconds


...

       Fetched 50 papers


2025-10-04 17:34:58,921 - arxiv - INFO - Sleeping: 2.436611 seconds


   Processing batch 4/10 (50 papers)


2025-10-04 17:35:01,363 - arxiv - INFO - Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=&id_list=2001.11314%2C2008.00623%2C1911.09070%2C2006.16668%2C2005.11401v4%2C2005.08100v1%2C2005.03191v3%2C2005.02593%2C2005.00700v3%2C2005.00700%2C1908.09791%2C2004.04136v4%2C2003.07845%2C2003.05997%2C2002.12530%2C2002.09402%2C2002.03184%2C2107.14795%2C2002.02925%2C2001.09977%2C1912.11370%2C1912.06680%2C1911.12385%2C1911.08460v3%2C1911.08265v2%2C1911.04252v4%2C1911.03864%2C1911.03894%2C1911.02116%2C1911.00172%2C1903.11059%2C1909.11556%2C1909.08053%2C1909.08053%2C1909.01792%2C1907.09109%2C1907.02544%2C1906.09777%2C1906.06832%2C1906.06423%2C1906.03805%2C1906.01702%2C1901.07291%2C1906.00091%2C1905.09272%2C1905.05513%2C1905.00546%2C1904.09408%2C1904.08378%2C1904.03819&sortBy=relevance&sortOrder=descending&start=0&max_results=100


KeyboardInterrupt: 

## Evaluation Functions

Functions to run classification and calculate evaluation metrics.

In [5]:
def evaluate_classifier(papers: List[Dict], ground_truth: Dict[str, int], 
                       confidence_threshold: float = 0.7, provider: str = 'deepseek') -> Dict:
    """
    Evaluate the ArXiv classifier against ground truth labels.
    
    Args:
        papers: List of paper dictionaries with metadata
        ground_truth: Dictionary mapping arxiv_id to label (1=positive, 0=negative)
        confidence_threshold: Confidence threshold for positive classification
        provider: LLM provider to use
        
    Returns:
        Dictionary with evaluation results
    """
    print(f" Running classification with {provider} (threshold={confidence_threshold})")
    
    # Initialize classifier
    classifier = ArXivClassifier(provider=provider, rate_limit_delay=CONFIG['rate_limit_delay'])
    
    # Run classification
    results = classifier.classify_papers(papers, confidence_threshold=confidence_threshold)
    
    # Extract predictions
    predictions = {}
    prediction_details = {}
    
    for classification in results['all_classifications']:
        arxiv_id = classification.get('arxiv_id')
        if arxiv_id in ground_truth:
            is_positive = classification.get('is_foundational_model', False)
            confidence = classification.get('confidence', 0.0)
            
            # Apply threshold
            prediction = 1 if (is_positive and confidence >= confidence_threshold) else 0
            predictions[arxiv_id] = prediction
            
            prediction_details[arxiv_id] = {
                'prediction': prediction,
                'confidence': confidence,
                'is_foundational': is_positive,
                'reasoning': classification.get('reasoning', ''),
                'ground_truth': ground_truth[arxiv_id]
            }
    
    # Calculate metrics
    y_true = [ground_truth[arxiv_id] for arxiv_id in predictions.keys()]
    y_pred = [predictions[arxiv_id] for arxiv_id in predictions.keys()]
    
    # Calculate all metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    evaluation_results = {
        'threshold': confidence_threshold,
        'provider': provider,
        'total_papers': len(predictions),
        'metrics': {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        },
        'confusion_matrix': {
            'true_positive': int(tp),
            'true_negative': int(tn),
            'false_positive': int(fp),
            'false_negative': int(fn)
        },
        'predictions': prediction_details,
        'classification_results': results
    }
    
    print(f" Evaluation complete:")
    print(f"   Accuracy: {accuracy:.3f}")
    print(f"   Precision: {precision:.3f}")
    print(f"   Recall: {recall:.3f}")
    print(f"   F1 Score: {f1:.3f}")
    
    return evaluation_results

In [6]:
def evaluate_multiple_thresholds(papers: List[Dict], ground_truth: Dict[str, int], 
                                thresholds: np.ndarray, provider: str = 'deepseek') -> List[Dict]:
    """
    Evaluate classifier performance across multiple confidence thresholds.
    
    Args:
        papers: List of paper dictionaries
        ground_truth: Ground truth labels
        thresholds: Array of confidence thresholds to test
        provider: LLM provider to use
        
    Returns:
        List of evaluation results for each threshold
    """
    print(f" Testing {len(thresholds)} different confidence thresholds...")
    
    # Run classification once to get all predictions
    classifier = ArXivClassifier(provider=provider, rate_limit_delay=CONFIG['rate_limit_delay'])
    results = classifier.classify_papers(papers, confidence_threshold=0.0)  # Get all predictions
    
    # Extract all predictions with confidence scores
    all_predictions = {}
    for classification in results['all_classifications']:
        arxiv_id = classification.get('arxiv_id')
        if arxiv_id in ground_truth:
            all_predictions[arxiv_id] = {
                'confidence': classification.get('confidence', 0.0),
                'is_foundational': classification.get('is_foundational_model', False),
                'ground_truth': ground_truth[arxiv_id]
            }
    
    # Test each threshold
    threshold_results = []
    
    for threshold in thresholds:
        print(f"   Testing threshold {threshold:.1f}...")
        
        # Apply threshold to existing predictions
        y_true = []
        y_pred = []
        
        for arxiv_id, pred_data in all_predictions.items():
            y_true.append(pred_data['ground_truth'])
            
            # Apply threshold
            prediction = 1 if (pred_data['is_foundational'] and 
                             pred_data['confidence'] >= threshold) else 0
            y_pred.append(prediction)
        
        # Calculate metrics
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        threshold_results.append({
            'threshold': threshold,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'true_positive': int(tp),
            'true_negative': int(tn),
            'false_positive': int(fp),
            'false_negative': int(fn)
        })
    
    print(" Threshold evaluation complete")
    return threshold_results

## Visualization Functions

Functions to create plots and visualizations of the evaluation results.

In [7]:
def plot_confusion_matrix(evaluation_results: Dict, save_path: str = None):
    """Plot confusion matrix from evaluation results."""
    cm = evaluation_results['confusion_matrix']
    
    # Create confusion matrix array
    confusion_array = np.array([
        [cm['true_negative'], cm['false_positive']],
        [cm['false_negative'], cm['true_positive']]
    ])
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_array, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted Negative', 'Predicted Positive'],
                yticklabels=['Actual Negative', 'Actual Positive'])
    
    plt.title(f"Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

def plot_threshold_comparison(threshold_results: List[Dict], save_path: str = None):
    """Plot metrics across different confidence thresholds."""
    df = pd.DataFrame(threshold_results)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # F1 Score
    axes[0, 0].plot(df['threshold'], df['f1_score'], 'o-', color='red', linewidth=2)
    axes[0, 0].set_title('F1 Score vs Confidence Threshold')
    axes[0, 0].set_xlabel('Confidence Threshold')
    axes[0, 0].set_ylabel('F1 Score')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Precision and Recall
    axes[0, 1].plot(df['threshold'], df['precision'], 'o-', label='Precision', linewidth=2)
    axes[0, 1].plot(df['threshold'], df['recall'], 's-', label='Recall', linewidth=2)
    axes[0, 1].set_title('Precision and Recall vs Confidence Threshold')
    axes[0, 1].set_xlabel('Confidence Threshold')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, 0].plot(df['threshold'], df['accuracy'], 'o-', color='green', linewidth=2)
    axes[1, 0].set_title('Accuracy vs Confidence Threshold')
    axes[1, 0].set_xlabel('Confidence Threshold')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Precision-Recall Curve
    axes[1, 1].plot(df['recall'], df['precision'], 'o-', linewidth=2)
    axes[1, 1].set_title('Precision-Recall Curve')
    axes[1, 1].set_xlabel('Recall')
    axes[1, 1].set_ylabel('Precision')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

def plot_confidence_distribution(evaluation_results: Dict, save_path: str = None):
    """Plot distribution of confidence scores by ground truth label."""
    predictions = evaluation_results['predictions']
    
    positive_confidences = [p['confidence'] for p in predictions.values() if p['ground_truth'] == 1]
    negative_confidences = [p['confidence'] for p in predictions.values() if p['ground_truth'] == 0]
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(positive_confidences, bins=20, alpha=0.7, label='Robotics Papers', color='green')
    plt.hist(negative_confidences, bins=20, alpha=0.7, label='Non-Robotics Papers', color='red')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.title('Confidence Score Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.boxplot([positive_confidences, negative_confidences], 
                labels=['Robotics', 'Non-Robotics'])
    plt.ylabel('Confidence Score')
    plt.title('Confidence Score by True Label')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

## Run Evaluation

Execute the evaluation pipeline on the loaded data.

In [8]:
# Main evaluation execution
if len(all_papers) > 0 and len(paper_labels) > 0:
    print(" Starting F1 Score Evaluation Pipeline")
    print("=" * 60)
    
    # 1. Single threshold evaluation
    print(f"\n Phase 1: Single Threshold Evaluation (threshold={CONFIG['confidence_threshold']})")
    evaluation_results = evaluate_classifier(
        papers=all_papers,
        ground_truth=paper_labels,
        confidence_threshold=CONFIG['confidence_threshold'],
        provider=CONFIG['provider']
    )
    
    # 2. Multiple threshold evaluation (if enabled)
    threshold_results = None
    if CONFIG['test_different_thresholds']:
        print(f"\n Phase 2: Multi-Threshold Evaluation")
        threshold_results = evaluate_multiple_thresholds(
            papers=all_papers,
            ground_truth=paper_labels,
            thresholds=CONFIG['threshold_range'],
            provider=CONFIG['provider']
        )
        
        # Find optimal threshold
        optimal_result = max(threshold_results, key=lambda x: x['f1_score'])
        print(f" Optimal threshold for F1 score: {optimal_result['threshold']:.1f}")
        print(f"   Best F1 Score: {optimal_result['f1_score']:.3f}")
    
    # 3. Save results
    if CONFIG['save_results']:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Save detailed results
        results_file = os.path.join(CONFIG['results_dir'], f'evaluation_results_{timestamp}.json')
        with open(results_file, 'w') as f:
            json.dump({
                'config': CONFIG,
                'evaluation_results': evaluation_results,
                'threshold_results': threshold_results
            }, f, indent=2, default=str)
        
        print(f" Results saved to: {results_file}")
    
    print(f"\n Evaluation Complete!")
    
else:
    print("️ No data available for evaluation.")
    print("   Please add your text files with arXiv IDs and run the data loading cells.")
    evaluation_results = None
    threshold_results = None

NameError: name 'all_papers' is not defined

## Results Visualization

Generate plots and visualizations of the evaluation results.

In [9]:
# Generate visualizations
if evaluation_results and CONFIG['plot_results']:
    print(" Generating visualizations...")
    
    # 1. Confusion Matrix
    print("   Creating confusion matrix...")
    plot_confusion_matrix(
        evaluation_results, 
        save_path=os.path.join(CONFIG['results_dir'], 'confusion_matrix.png') if CONFIG['save_results'] else None
    )
    
    # 2. Confidence Distribution
    print("   Creating confidence distribution plot...")
    plot_confidence_distribution(
        evaluation_results,
        save_path=os.path.join(CONFIG['results_dir'], 'confidence_distribution.png') if CONFIG['save_results'] else None
    )
    
    # 3. Threshold Comparison (if available)
    if threshold_results:
        print("   Creating threshold comparison plots...")
        plot_threshold_comparison(
            threshold_results,
            save_path=os.path.join(CONFIG['results_dir'], 'threshold_comparison.png') if CONFIG['save_results'] else None
        )
    
    print(" Visualizations complete!")

elif evaluation_results:
    print(" Plotting disabled in configuration")
else:
    print("️ No results available for visualization")

NameError: name 'evaluation_results' is not defined

## Detailed Results Analysis

Analyze the results in detail, including error analysis and specific misclassifications.

In [ ]:
# Detailed analysis of results
if evaluation_results:
    print(" Detailed Results Analysis")
    print("=" * 60)
    
    # Summary metrics
    metrics = evaluation_results['metrics']
    cm = evaluation_results['confusion_matrix']
    
    print(f"\n Performance Summary:")
    print(f"   Total Papers Evaluated: {evaluation_results['total_papers']}")
    print(f"   Confidence Threshold: {evaluation_results['threshold']:.1f}")
    print(f"   Provider: {evaluation_results['provider'].upper()}")
    print(f"\n    Metrics:")
    print(f"      Accuracy:  {metrics['accuracy']:.3f}")
    print(f"      Precision: {metrics['precision']:.3f}")
    print(f"      Recall:    {metrics['recall']:.3f}")
    print(f"      F1 Score:  {metrics['f1_score']:.3f}")
    
    print(f"\n    Confusion Matrix:")
    print(f"      True Positives:  {cm['true_positive']} (correctly identified robotics papers)")
    print(f"      True Negatives:  {cm['true_negative']} (correctly identified non-robotics papers)")
    print(f"      False Positives: {cm['false_positive']} (non-robotics misclassified as robotics)")
    print(f"      False Negatives: {cm['false_negative']} (robotics misclassified as non-robotics)")
    
    # Error analysis
    predictions = evaluation_results['predictions']
    
    # False Positives (Type I errors)
    false_positives = [
        (arxiv_id, pred) for arxiv_id, pred in predictions.items()
        if pred['ground_truth'] == 0 and pred['prediction'] == 1
    ]
    
    # False Negatives (Type II errors)
    false_negatives = [
        (arxiv_id, pred) for arxiv_id, pred in predictions.items()
        if pred['ground_truth'] == 1 and pred['prediction'] == 0
    ]
    
    print(f"\n Error Analysis:")
    
    if false_positives:
        print(f"\n   False Positives ({len(false_positives)} papers):")
        print("   (Non-robotics papers incorrectly classified as robotics)")
        for i, (arxiv_id, pred) in enumerate(false_positives[:5]):  # Show first 5
            print(f"      {i+1}. {arxiv_id} (confidence: {pred['confidence']:.2f})")
            print(f"         Reasoning: {pred['reasoning'][:100]}...")
        if len(false_positives) > 5:
            print(f"         ... and {len(false_positives) - 5} more")
    
    if false_negatives:
        print(f"\n   False Negatives ({len(false_negatives)} papers):")
        print("   (Robotics papers incorrectly classified as non-robotics)")
        for i, (arxiv_id, pred) in enumerate(false_negatives[:5]):  # Show first 5
            print(f"      {i+1}. {arxiv_id} (confidence: {pred['confidence']:.2f})")
            print(f"         Reasoning: {pred['reasoning'][:100]}...")
        if len(false_negatives) > 5:
            print(f"         ... and {len(false_negatives) - 5} more")
    
    # Confidence analysis
    all_confidences = [pred['confidence'] for pred in predictions.values()]
    positive_confidences = [pred['confidence'] for pred in predictions.values() if pred['ground_truth'] == 1]
    negative_confidences = [pred['confidence'] for pred in predictions.values() if pred['ground_truth'] == 0]
    
    print(f"\n Confidence Score Analysis:")
    print(f"   Overall confidence range: {min(all_confidences):.2f} - {max(all_confidences):.2f}")
    print(f"   Average confidence - Robotics papers: {np.mean(positive_confidences):.2f}")
    print(f"   Average confidence - Non-robotics papers: {np.mean(negative_confidences):.2f}")
    
    # Threshold analysis summary
    if threshold_results:
        best_f1 = max(threshold_results, key=lambda x: x['f1_score'])
        best_accuracy = max(threshold_results, key=lambda x: x['accuracy'])
        
        print(f"\n Threshold Optimization Results:")
        print(f"   Best F1 Score: {best_f1['f1_score']:.3f} at threshold {best_f1['threshold']:.1f}")
        print(f"   Best Accuracy: {best_accuracy['accuracy']:.3f} at threshold {best_accuracy['threshold']:.1f}")
        print(f"   Current threshold ({CONFIG['confidence_threshold']:.1f}) F1: {metrics['f1_score']:.3f}")
        
        if best_f1['threshold'] != CONFIG['confidence_threshold']:
            improvement = best_f1['f1_score'] - metrics['f1_score']
            print(f"   Potential F1 improvement: +{improvement:.3f} by using threshold {best_f1['threshold']:.1f}")

else:
    print("️ No evaluation results available for analysis")

## Summary and Next Steps

### What this notebook does:

1. **Data Loading**: Loads ground truth arXiv IDs from text files
2. **Metadata Fetching**: Retrieves paper titles and abstracts from arXiv API
3. **Classification**: Runs your ArXiv filter on all papers
4. **Evaluation**: Calculates F1 score, precision, recall, and accuracy
5. **Analysis**: Provides detailed error analysis and optimization suggestions
6. **Visualization**: Creates plots showing performance across different thresholds

### To use this notebook:

1. **Add your data files**:
   - `robotics_arxiv_ids.txt`: One arXiv ID per line for robotics papers
   - `non_robotics_arxiv_ids.txt`: One arXiv ID per line for non-robotics papers

2. **Run all cells** to execute the complete evaluation pipeline

3. **Review results** in the detailed analysis section

### Key metrics explained:

- **F1 Score**: Harmonic mean of precision and recall (balanced measure)
- **Precision**: Of papers classified as robotics, how many actually are? (False positive control)
- **Recall**: Of actual robotics papers, how many did we find? (False negative control)
- **Accuracy**: Overall percentage of correct classifications

### Files you need to create:

```
robotics_arxiv_ids.txt          # Positive examples
non_robotics_arxiv_ids.txt      # Negative examples
```

In [ ]:
# This cell is ready for your custom analysis or additional experiments
# You can use this space to:
# - Test different providers (deepseek vs gemini)
# - Analyze specific subsets of papers
# - Export results in different formats
# - Add custom visualizations

print(" Ready for additional analysis!")